# Text Similarity (STS) – Notebook 2: Pairwise synset similarity aggregation

## Project Overview

This notebook is part of a research project devoted to the **estimation of semantic textual similarity (STS)** using lexical-semantic resources and vector-based representations. The objective is to implement and compare several approaches for computing sentence-level semantic similarity by **combining standard lexical similarity measures with different aggregation strategies at the sentence level**. Four alternative aggregation strategies are explored in this project. Each approach is implemented in a different notebook:

1) **[Notebook 1](PLN-Proyecto%20-%20PARTE1%20-%20Liw-Wang%20con%20funciones%20wordnet.ipynb): Cosine similarity between semantic vectors derived from lexical similarity scores**  as described in [Liu & Wang 2014]
2) **Notebook 2: A pairwise synset similarity aggregation strategy** (proposed in this notebook)
3) **[Notebook 3](PLN-Proyecto%20-%20PARTE3%20-%20Agirre%20con%20funciones%20wordnet.ipynb) Overlap similarity**  (based on the approach described in [Agirre et al. 2017](https://aclanthology.org/S17-2001/))

All notebooks in this project share the same experimental pipeline. The workflow includes **text preprocessing, synset assignment using WordNet (when applicable), computation of lexical similarity between tokens, and aggregation of these similarities into sentence-level similarity scores**. The notebooks differ in the formulation of the **sentence similarity stage**, while the preceding components remain identical in order to ensure comparability across experiments.

The evaluation is performed using sentence pairs from the **STSBenchmark** dataset, which provides human similarity judgments for pairs of English sentences. System scores produced by the implemented methods are compared with the gold-standard annotations using **Pearson correlation**.

Intermediate results and similarity scores generated by the notebooks were exported and analyzed externally. A summary of the experimental results is available in the [project report](../Procedimiento%20y%20discusión.pdf). Final analysis can be consulted in the [experiments comparison results file](../comparativa.xlsx).

## Subproject Overview – Pairwise synset similarity aggregation

This notebook explores a sentence similarity strategy based on **direct aggregation of pairwise lexical similarities between synsets**. After preprocessing the sentences and assigning WordNet synsets to the tokens, similarity between two sentences is computed by comparing **every synset in the first sentence with every synset in the second**.

For each sentence pair, a **matrix of lexical similarity scores** is constructed. Each element of the matrix represents the similarity between one synset from the first sentence and one synset from the second. Lexical similarity between synsets is computed using several **WordNet-based similarity measures**.

The final sentence-level similarity score is obtained by **aggregating the values of the similarity matrix**. In this implementation, the score corresponds to the **mean of all pairwise synset similarity values**.


## Implementation

### Experiment configuration, dependencies, and auxiliary functions

In the following cells we set the **global configuration of the experiment** and load the required libraries. The main experimental parameters controlling the execution of the notebook are defined here. These include whether **Lesk-based word sense disambiguation** is applied, the type of **bag of words / bag of synsets** considered in the calculations, the **sentence-level similarity method** to be used, and the **corpus partition** to be processed.

Additional Boolean variables control whether libraries and data should be imported during execution. A descriptive string (`criterios`) is also created to identify the configuration of each experimental run when results are exported to an Excel file.

In [ ]:
# Parámetros a definir

    # lesk: True/False
        # True --> Synset se elige utilizando el desambiguador de Lesk
        # False --> Synset es el más común 
    # tipo_bow: --> Categorías de synsets que cogeremos para hacer los cálculos
        # todos (1) --> Todos los synsets
        # nv (2) --> Solo nombres y verbos
    # metodo_frase: lw / Ariadna / Agirre
    # partición: numero entero
        # fragmento del corpus que se procesa.
        

desambiguar = True

expandir_contracciones = False

tipo_bow = 'nv'

metodo_frase = 'Ariadna'

particion = 1

print_info = False

importar_librerias = True

importar_datos = True

criterios = 'Lesk-' + str(desambiguar) + ', tipo_bow-' + str(tipo_bow) + ', metodo-' + str(metodo_frase) + ', particion-' + str(particion)

print()
print('Desambiguación desambiguar:', desambiguar)
print('Tipo de BoW:', tipo_bow)
print('Método de comparación a nivel de frase:', metodo_frase)
print('Particion de filas a importar:', particion)
print('Importar librerías:', importar_librerias)
print('Importar datos:', importar_datos)

print(criterios)



In [ ]:
# Importación de librerías

if importar_librerias:
    
    import time
    import datetime

    import csv

    import pandas as pd
    pd.options.display.max_rows  # Para mostrar todas las columnas
    pd.set_option('display.max_colwidth', None)  # Para incluir todo el contenido de una celda, sin truncar contenido.
    pd.set_option('display.max_columns', 500)  # Para incluir todas las columnas (no serán más de 500)
    pd.set_option('display.max_rows', 6000)  # Para incluir todas las filas (no serán más de 6000)
    
    import nltk

    def ensure_nltk_resource(resource_path, download_name):
        try:
            nltk.data.find(resource_path)
        except LookupError:
            nltk.download(download_name)

    # Recursos necesarios de NLTK
    ensure_nltk_resource("corpora/wordnet", "wordnet")
    ensure_nltk_resource("corpora/omw-1.4", "omw-1.4")
    ensure_nltk_resource("corpora/wordnet_ic", "wordnet_ic")
    ensure_nltk_resource("corpora/genesis", "genesis")

    from nltk.corpus import wordnet as wn

    from nltk.corpus import wordnet_ic
    ic_brown = wordnet_ic.ic('ic-brown.dat')
    ic_semcor = wordnet_ic.ic('ic-semcor.dat')

    from nltk.corpus import genesis
    ic_genesis = wn.ic(genesis, False, 0.0)
    
    if desambiguar == True:
        from nltk.wsd import lesk

import numpy as np

### Dataset import, cleaning and reduced corpus creation

The **STSBenchmark training dataset** is loaded into a pandas dataframe. The file contains pairs of English sentences annotated with **human similarity scores**, which serve as the reference values for evaluation. Duplicated sentence pairs are removed to avoid counting identical examples more than once during the experiments.

A reduced dataframe (`c`) is then created by selecting rows from the original corpus at regular intervals. This subset is used during development to **test preprocessing and similarity algorithms more quickly**, allowing faster iteration before running the full experiment on the complete dataset.

In [ ]:
# Importación de los datos del fichero sts-train.csv a un dataframe

start_total = time.time()

if importar_datos:

    file = '../stsbenchmark/sts-train.csv'
    corpus = pd.read_csv(file, sep='\t', on_bad_lines='skip', quoting=csv.QUOTE_NONE, usecols=range(7), header=None)
    corpus.head()
    corpus.columns = ['genre', 'file', 'year', 'id', 'gold_score', 'sent1', 'sent2']


end_total = time.time()

print('TIEMPO TRANSCURRIDO:', end_total-start_total)

corpus.sample(5)

In [ ]:
# Eliminación de duplicados

if importar_datos:

    corpus.drop_duplicates(['sent1', 'sent2'], keep='first', inplace=True)

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

In [ ]:
# Creación de un corpus c reducido con filas representativas del corpus original

    # Para realizar pre-procesamientos que nos permitan verificar la adecuación de los algoritmos
    # en un tiempo reducido.

particion = particion

c = corpus.iloc[::particion, :].copy()
num_filas = 'Número de filas incluídas en el dataframe: ' + str(c.shape[0])

print(num_filas)

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

c.head(12)

### Text preprocessing

Each sentence in the dataset undergoes the following standard natural language processing steps:  
1) tokenization  
2) lowercase conversion  
3) part-of-speech tagging*  
4) stopword removal (to retain only content-bearing words).

The resulting tokens are then prepared for the subsequent stages of the pipeline. The preprocessing results are stored in additional columns of the dataframe so that both the original sentences and their processed representations remain available.

A method for expanding contractions is implemented but not currently used (kept for potential future improvements).

In [ ]:
# Pre procesamiento

# ========================================
# 3. TOKENIZACION, LIMPIEZA Y ANOTACIÓN
# ========================================

import nltk

def expandir_contracciones_texto(texto):
    # specific
    texto = re.sub(r"won\'t", "will not", texto)
    texto = re.sub(r"can\'t", "can not", texto)

    # general
    texto = re.sub(r"n\'t", " not", texto)
    texto = re.sub(r"\'re", " are", texto)
    texto = re.sub(r"\'s", " is", texto)
    texto = re.sub(r"\'d", " would", texto)
    texto = re.sub(r"\'ll", " will", texto)
    texto = re.sub(r"\'t", " not", texto)
    texto = re.sub(r"\'ve", " have", texto)
    texto = re.sub(r"\'m", " am", texto)
    return texto

def tokenizar(sent, print_tokenizar =False):
    
    # tokens que aceptaremos (de nltk.org/book/ch03). Output: cadena de palabras.
   
    if print_tokenizar == True: print('Frase a tokenizar:', sent)
        
    pattern = r'''(?x)       # set flag to allow verbose regexps
        (?:[A-Z]\.)+         # abbreviations, e.g. U.S.A. ?: needs to be added to specify that the parenthesis specify the scope of the disjunction, not the selection o strings to be extracted
       | \w+(?:-\w+)*        # words with optional internal hyphens
       | [\$,\€]?\d+(?:\.\d+)?%?  # currency and percentages, e.g. $12.40, 82%
    '''
    if print_tokenizar == True: print('pattern:', pattern)
    '''
    AMPLIAR.
    - Analizar errores
    - Alguna solución para detectar phrasal verbs?
    '''

    # tokenización. Output: lista
   
    tokens = nltk.regexp_tokenize(sent, pattern, gaps=False)
    
    # limpieza 1: mayúsculas
    
    tokens = [token.lower() for token in tokens]   
    if print_tokenizar == True: print('Tokens:', tokens)
 
    # anotación POS. Output: lista de tuplas
    
    tagged_tokens = nltk.pos_tag(tokens, tagset='universal')
    
    # Convertimos las etiquetas gramaticales en notación Lesk;
        
    for i, tagged_token in enumerate(tagged_tokens):
        if tagged_tokens[i][1] == 'NOUN': tagged_tokens[i] = (tagged_tokens[i][0], 'n')
        if tagged_tokens[i][1] == 'VERB': tagged_tokens[i] = (tagged_tokens[i][0], 'v')
        if tagged_tokens[i][1] == 'ADJ': tagged_tokens[i] = (tagged_tokens[i][0], 'a')
        if tagged_tokens[i][1] == 'ADV': tagged_tokens[i] = (tagged_tokens[i][0], 'r')
    if print_tokenizar == True: print('Tagged_tokens:', tagged_tokens)
    
    # limipieza 2: stopwords - REVISAR si queremos/podemos incluir phrasal verbs, determinantes, pronombres...
            
                # Wordnet no incluye determinantes o preposiciones, de forma que "a" es asociado por Lesk 
                # al Synset('deoxyadenosine_monophosphate.n.01'), lo cual no nos conviene.
                # Por otro lado, Lesk utiliza para desambiguar solo las palabras que aparecen en Wordnet,
                # de modo que limpiar antes o después de Lesk las stopwords no debería afectar al resultado.
   
    if print_tokenizar == True: print('Limpieza de stopwords')
    stopwords = nltk.corpus.stopwords.words('english')
    tagged_stops = [tagged_stop for tagged_stop in tagged_tokens if tagged_stop[0] in stopwords]
    if print_tokenizar == True: print('Stopwords eliminadas:', tagged_stops)
    tokens = [token for token in tokens if token not in stopwords]
    tagged_tokens = [tagged_token for tagged_token in tagged_tokens if tagged_token[0] not in stopwords]
    if print_tokenizar == True: print('Tagged tokens sin stopwords:', tagged_tokens)

    return(tokens, tagged_tokens, tagged_stops)

# Ejemplo para comprobar funcionamiento tokenizar()
   
sent = "I'll always love my sweet baby"

print('****** EJEMPLO tokenizar() ********')
tokens = tokenizar(sent, print_tokenizar=True)
print('****')

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)


In [ ]:
c.head()

In [ ]:
# Tokenización de c

c['tokens1'], c['tokens2'] = None, None
c['tagged_tokens1'], c['tagged_tokens2'] = None, None
c['stops1'], c['stops2'] = None, None

print_tokenizar = False

for i in c.index:
    tok1 = tokenizar(c.at[i, 'sent1'], print_tokenizar)
    tok2 = tokenizar(c.at[i, 'sent2'], print_tokenizar)

    c.at[i, 'tokens1'] = tok1[0]
    c.at[i, 'tokens2'] = tok2[0]
    c.at[i, 'tagged_tokens1'] = tok1[1]
    c.at[i, 'tagged_tokens2'] = tok2[1]
    c.at[i, 'stops1'] = tok1[2]
    c.at[i, 'stops2'] = tok2[2]

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

c.head()

### Synset extraction

Preprocessed tokens are associated with **WordNet synsets** when possible. When enabled, **Lesk-based word sense disambiguation** is applied to select the most appropriate synset for each token; otherwise the first synset provided by WordNet is used. Tokens for which no synset is found are recorded separately.


In [ ]:
# ========================================
# 4. ASIGNACIÓN DE SYNSETS DE WORDNET
# ========================================

# Asignación de synsets 
    # Entrada: lista de tuplas con tokens y sus correspondientes etiquetas gramaticales
    # Argumentos: 
         # lesk: Opción de aplicar el algoritmo de Lesk  [Lesk 1986] de la librería wsd para desambiguar
                # Opción por defecto: n (no) --> Se asignará el primer synset del grupo de synsets (el más común)
                # Opción: s (sí) --> Asignar el synset determinado por Lesk
                
def syns(tagged_tokens, desambiguar=None):
    from nltk.wsd import lesk

    syns = list()
    errors = list()
    if desambiguar == True:
        for j in tagged_tokens:
            resultado_lesk = lesk([token for token, tag in tagged_tokens], j[0], j[1])
            if resultado_lesk is not None:
                syns.append(resultado_lesk)
            else:
                errors.append(j)
    else: 
        for j in tagged_tokens:
            try: syns.append(wn.synsets(j[0],j[1])[0])
            except: errors.append(j)
    return(syns, errors)
    
# Ejemplo para comprobar funcionamiento syns()

desambiguar_test = True

print('Desambiguación Lesk:', desambiguar_test )

tagged_tokens = c['tagged_tokens1'][151]
syns1 = syns(tagged_tokens, desambiguar_test )

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

print('****** EJEMPLO syns() ********')   
print('tagged_tokens:', tagged_tokens)
print('syns(tagged_tokens):', syns1[0])
print('syns(tagged_tokens) - Errors:', syns1[1])
print('****')

desambiguar_test = False

print('Desambiguación Lesk:', desambiguar_test )

tagged_tokens = c['tagged_tokens1'][151]
syns1 = syns(tagged_tokens, desambiguar_test )

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

print('****** EJEMPLO syns() ********')   
print('tagged_tokens:', tagged_tokens)
print('syns(tagged_tokens):', syns1[0])
print('syns(tagged_tokens) - Errors:', syns1[1])
print('****')

In [ ]:
# Asignacion de synsets a los tokens de c

desambiguar = desambiguar
print('Desambiguación Lesk:', desambiguar)
tipo_bow = tipo_bow
print('Tipo_bow:', tipo_bow)

c['syns1'] = None
c['syns2'] = None
c['errors1'] = None
c['errors2'] = None
c['syns1_nv'] = None
c['syns2_nv'] = None
c['syns1_resto'] = None
c['syns2_resto'] = None

for i in c.index:

    res1 = syns(c.at[i, 'tagged_tokens1'], desambiguar)
    res2 = syns(c.at[i, 'tagged_tokens2'], desambiguar)

    c.at[i,'syns1'] = res1[0]
    c.at[i,'syns2'] = res2[0]
    c.at[i,'errors1'] = res1[1]
    c.at[i,'errors2'] = res2[1]

    if tipo_bow == 'nv':
        c.at[i,'syns1_nv'] = [syn for syn in res1[0] if syn.pos() in ['n','v']]
        c.at[i,'syns2_nv'] = [syn for syn in res2[0] if syn.pos() in ['n','v']]
        c.at[i,'syns1_resto'] = [syn for syn in res1[0] if syn.pos() not in ['n','v']]
        c.at[i,'syns2_resto'] = [syn for syn in res2[0] if syn.pos() not in ['n','v']]

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

c.sample(5)

### Bag of words / bag of synsets construction

A **bag of words (BoW)** is constructed for each sentence pair by combining the lexical elements extracted from both sentences. In practice, this representation corresponds to a **bag of synsets**, which defines the **vector space used in the sentence similarity calculations**.

Depending on the experimental configuration, the representation may include **all synsets** or only those corresponding to **nouns and verbs**. Duplicate elements are removed to obtain a set of unique lexical items.

In [ ]:
# ========================================
# 5. BAG OF WORDS
# ========================================

# Creación de la Bag of Words (BoW) (de hecho, será una "Bag of synsets")
    # La necesitaremos luego como espacio vectorial para aplicar el método [Liu 2013] de similitud a nivel de frase.

print('Tipo_bow:', tipo_bow)
    
c['bow'] = c['syns1'] + c['syns2']
c['bow_errors'] =  c['errors1'] + c['errors2']

if tipo_bow == 'nv':
    c['bow_nv'] =  c['syns1_nv'] + c['syns2_nv']
    c['bow_resto'] =  c['syns1_resto'] + c['syns2_resto']

for i in c.index:
    c.at[i, 'bow'] = list(set(c.at[i, 'bow']))
    c.at[i, 'bow_errors'] = list(set(c.at[i, 'bow_errors']))    
    if tipo_bow == 'nv':
        c.at[i, 'bow_nv'] = list(set(c.at[i, 'bow_nv']))
        c.at[i, 'bow_resto'] = list(set(c.at[i, 'bow_resto']))

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

c.sample(5)

### Sentence vector construction

A method for sentence representation is defined using the **bag of synsets defined previously as the vector space**. For each dimension of this space, the value of the sentence vector corresponds to the **maximum lexical similarity between the associated synset and the synsets present in the sentence**. Lexical similarity between synsets is computed using different **WordNet-based similarity measures**.

Sentence vectors will later be computed according to this method when sentence similarity is evaluated.

In [ ]:
# ========================================
# 6. SIMILITUD A NIVEL LÉXICO - MÉTODOS INCLUÍDOS EN WORDNET
# ========================================

# Ejemplos de funciones de similitud integradas en Wordnet

    # ps --> path_similarity (POR DEFECTO). Rango: [0:1]
    # lch --> Leacock Chodorow. Rango: [0:3.64?] 
    # wup --> Wu_Palmer
    # res --> Resknik (requiere corpus ic)
    # jcn --> Jiang-Conrath (requiere corpus ic)
    # lin --> Lin (requiere corpus ic)

# Ejemplos para comprobar funcionamiento de los distintos métodos de simlitud de wordnet
    
syn1 = wn.synset('dog.n.01')
syn2 = wn.synset('cat.n.01')
syn3 = wn.synset('black.a.01')
syn4 = wn.synset('white.a.01')

try: print('wn.path_similarity(', syn1, syn1, '):',  wn.path_similarity(syn1, syn1))
except Exception as e: print('ERROR', e)
try: print('wn.path_similarity(', syn1, syn2, '):',  wn.path_similarity(syn1, syn2))
except Exception as e: print('ERROR', e)
try: print('wn.path_similarity(', syn1, syn3, '):',  wn.path_similarity(syn1, syn3))
except Exception as e: print('ERROR', e)
try: print('wn.path_similarity(', syn3, syn4, '):',  wn.path_similarity(syn3, syn4))
except Exception as e: print('ERROR', e)
    
print()

try: print('wn.lch_similarity(', syn1, syn1, '):',  wn.lch_similarity(syn1, syn1))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn2, '):',  wn.lch_similarity(syn1, syn2))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn3, '):',  wn.lch_similarity(syn1, syn3))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn3, syn4, '):',  wn.lch_similarity(syn3, syn4))
except Exception as e: print('ERROR', e)    

print()

try: print('wn.wup_similarity(', syn1, syn1, '):',  wn.wup_similarity(syn1, syn1))
except Exception as e: print('ERROR', e)
try: print('wn.wup_similarity(', syn1, syn2, '):',  wn.wup_similarity(syn1, syn2))
except Exception as e: print('ERROR', e)
try: print('wn.wup_similarity(', syn1, syn3, '):',  wn.wup_similarity(syn1, syn3))
except Exception as e: print('ERROR', e)
try: print('wn.wup_similarity(', syn3, syn4, '):',  wn.wup_similarity(syn3, syn4))
except Exception as e: print('ERROR', e)    

print()

ic = ic_brown
print('ic_brown')
try: print('wn.lch_similarity(', syn1, syn1, '):',  wn.res_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn2, '):',  wn.res_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn3, '):',  wn.res_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn3, syn4, '):',  wn.res_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

try: print('wn.jcn_similarity(', syn1, syn1, '):',  wn.jcn_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn1, syn2, '):',  wn.jcn_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn1, syn3, '):',  wn.jcn_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn3, syn4, '):',  wn.jcn_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

try: print('wn.lin_similarity(', syn1, syn1, '):',  wn.lin_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn1, syn2, '):',  wn.lin_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn1, syn3, '):',  wn.lin_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn3, syn4, '):',  wn.lin_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

ic = ic_semcor
print('ic_semcor')
try: print('wn.lch_similarity(', syn1, syn1, '):',  wn.res_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn2, '):',  wn.res_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn3, '):',  wn.res_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn3, syn4, '):',  wn.res_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

try: print('wn.jcn_similarity(', syn1, syn1, '):',  wn.jcn_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn1, syn2, '):',  wn.jcn_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn1, syn3, '):',  wn.jcn_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn3, syn4, '):',  wn.jcn_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

try: print('wn.lin_similarity(', syn1, syn1, '):',  wn.lin_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn1, syn2, '):',  wn.lin_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn1, syn3, '):',  wn.lin_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn3, syn4, '):',  wn.lin_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

ic = ic_genesis
print('ic_genesis')
try: print('wn.lch_similarity(', syn1, syn1, '):',  wn.res_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn2, '):',  wn.res_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn1, syn3, '):',  wn.res_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lch_similarity(', syn3, syn4, '):',  wn.res_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

try: print('wn.jcn_similarity(', syn1, syn1, '):',  wn.jcn_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn1, syn2, '):',  wn.jcn_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn1, syn3, '):',  wn.jcn_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.jcn_similarity(', syn3, syn4, '):',  wn.jcn_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

try: print('wn.lin_similarity(', syn1, syn1, '):',  wn.lin_similarity(syn1, syn1, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn1, syn2, '):',  wn.lin_similarity(syn1, syn2, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn1, syn3, '):',  wn.lin_similarity(syn1, syn3, ic))
except Exception as e: print('ERROR', e)
try: print('wn.lin_similarity(', syn3, syn4, '):',  wn.lin_similarity(syn3, syn4, ic))
except Exception as e: print('ERROR', e)

print()

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

### Sentence similarity computation – Pairwise synset similarity aggregation

Sentence similarity is computed using a **pairwise synset comparison approach**. For each sentence pair, a **matrix of lexical similarities** is built between the synsets of the two sentences, where each cell represents the similarity between one synset from the first sentence and one from the second. Lexical similarity is computed using several **WordNet-based measures**. For IC-based measures (`res`, `jcn`, `lin`), the computation is repeated using different information-content corpora.

The final sentence-level score is obtained as the **mean of all values in the similarity matrix**.

In [ ]:
# ========================================
# SIMILITUD A NIVEL DE FRASE - ARIADNA
# ========================================

# Matriz de similitud entre synsets
#   Entrada:
#     - syns1, syns2: listas de synsets correspondientes a dos frases
#   Argumentos:
#   - medida: función de similitud léxica que se quiere aplicar
#       - ps  --> path_similarity
#           Rango: [0,1]
#       - lch --> Leacock-Chodorow
#           Rango aproximado: [0,3.64]
#       - wup --> Wu-Palmer
#       - res --> Resnik (requiere corpus IC)
#           - mismo synset --> depende del IC del synset
#           - si la palabra no está en el corpus, la similitud puede ser infinita
#             MODIFICACIÓN: se asigna el valor 9.04
#       - jcn --> Jiang-Conrath (requiere corpus IC)
#           - mismo synset --> similitud infinita
#             MODIFICACIÓN: se asigna similitud = 1
#       - lin --> Lin (requiere corpus IC)
#   - ic: corpus en formato Information Content, para las funciones que lo requieren
#   - print_similitud: si es True, se imprimirá información sobre el procesamiento
# Requisitos:
#   - Haber importado wordnet_ic si se utilizan res, jcn o lin
# Salida:
#   - matriz de dimensión len(syns1) x len(syns2), donde cada valor [i,j]
#     representa la similitud entre syns1[i] y syns2[j]

    
def matriz_similitud(syns1, syns2, medida='ps', ic = '', print_info=False):
    
    s = np.zeros(shape = (len(syns1), len(syns2)))    # similitud entre los syns
    
    if print_info == True: 
    
        print('syns1:', syns1)
        print('syns2:', syns2)
        print()
        print('Cálculo de la matriz similitud de syns1 y syns2')
        print()    
        print('Método de similitud léxica:', medida)
        print()       
        
    for i in range(len(syns1)):
        for j in range(len(syns2)):
            sim = 0.0
            if (syns1[i].pos() == syns2[j].pos()):
                if medida == 'ps':
                    try: sim = wn.path_similarity(syns1[i], syns2[j])
                    except Exception as e:
                        if print_info: print('ERROR', e)
                elif medida == 'lch':
                    try: sim = wn.lch_similarity(syns1[i], syns2[j])
                    except Exception as e:
                        if print_info: print('ERROR', e)
                elif medida == 'wup':
                    try: sim = wn.wup_similarity(syns1[i], syns2[j])
                    except Exception as e:
                        if print_info: print('ERROR', e)
                elif medida == 'res':
                    try: sim = wn.res_similarity(syns1[i], syns2[j], ic)
                    except Exception as e:
                        if print_info: print('ERROR', e)
                        sim = 0                    
                    if sim >= 1.e+10:
                        sim = 9.04
                elif medida == 'jcn':  
                    if syns1[i] == syns2[j]:
                        sim = 1.                        
                    else:
                        try: sim = wn.jcn_similarity(syns1[i], syns2[j], ic)
                        except Exception as e:
                            if print_info: print('ERROR', e)
                    if sim >= 1.e+10:
                        sim = 1
                elif medida == 'lin':
                    try: sim = wn.lin_similarity(syns1[i], syns2[j], ic)
                    except Exception as e:
                        if print_info: print('ERROR', e)
                if sim is None:
                    continue
                else:
                    s[i,j] = sim
    return(s)


# Ejemplo para comprobar funcionamiento matriz_similitud()

syns1 = c['syns1'][501]
syns2 = c['syns2'][501]

print_info = True

ic_brown = wordnet_ic.ic('ic-brown.dat')

print('****** EJEMPLO matriz_similitud() ********')   
print('syns1:', syns1)
print('syns2:', syns2)
print('matriz_similitud(syns1, syns2, \'ps\'):')
print(matriz_similitud(syns1, syns2, 'ps', print_info=print_info))
print('matriz_similitud(syns1, syns2, \'lch\'):')
print(matriz_similitud(syns1, syns2, 'lch', print_info=print_info))
print('matriz_similitud(syns1, syns2, \'wup\'):')
print(matriz_similitud(syns1, syns2, 'wup', print_info=print_info))
print('matriz_similitud(syns1, syns2, \'res\', ic_brown):')
print(matriz_similitud(syns1, syns2, 'res', ic_brown, print_info=print_info))
print('matriz_similitud(syns1, syns2, \'jcn\', ic_brown):')
print(matriz_similitud(syns1, syns2, 'jcn', ic_brown, print_info=print_info))
print('matriz_similitud(syns1, syns2, \'lin\', ic_brown):')
print(matriz_similitud(syns1, syns2, 'lin', ic_brown, print_info=print_info))
print('****')


In [ ]:
# Creación y almacenamiento de las matrices de similitud


if metodo_frase == 'Ariadna':

    c['matriz_sim_ps'] = None
    c['matriz_sim_lch'] = None
    c['matriz_sim_wup'] = None
    c['matriz_sim_res_brown'] = None
    c['matriz_sim_res_semcor'] = None
    c['matriz_sim_res_genesis'] = None
    c['matriz_sim_jcn_brown'] = None
    c['matriz_sim_jcn_semcor'] = None
    c['matriz_sim_jcn_genesis'] = None
    c['matriz_sim_lin_brown'] = None
    c['matriz_sim_lin_semcor'] = None
    c['matriz_sim_lin_genesis'] = None

    print('Tipo_bow:', tipo_bow)

    for i in c.index:

        if tipo_bow == 'nv':
            syns1 = c.at[i, 'syns1_nv']
            syns2 = c.at[i, 'syns2_nv']
        else:
            syns1 = c.at[i, 'syns1']
            syns2 = c.at[i, 'syns2']        
        c.at[i, 'matriz_sim_ps'] = matriz_similitud(syns1, syns2, 'ps')
        c.at[i, 'matriz_sim_lch'] = matriz_similitud(syns1, syns2, 'lch')
        c.at[i, 'matriz_sim_wup'] = matriz_similitud(syns1, syns2, 'wup')

        ic = ic_brown

        c.at[i, 'matriz_sim_res_brown'] = matriz_similitud(syns1, syns2, 'res', ic)    
        c.at[i, 'matriz_sim_jcn_brown'] = matriz_similitud(syns1, syns2, 'jcn', ic)    
        c.at[i, 'matriz_sim_lin_brown'] = matriz_similitud(syns1, syns2, 'lin', ic)

        ic = ic_semcor

        c.at[i, 'matriz_sim_res_semcor'] = matriz_similitud(syns1, syns2, 'res', ic)    
        c.at[i, 'matriz_sim_jcn_semcor'] = matriz_similitud(syns1, syns2, 'jcn', ic)    
        c.at[i, 'matriz_sim_lin_semcor'] = matriz_similitud(syns1, syns2, 'lin', ic)

        ic = ic_genesis

        c.at[i, 'matriz_sim_res_genesis'] = matriz_similitud(syns1, syns2, 'res', ic)    
        c.at[i, 'matriz_sim_jcn_genesis'] = matriz_similitud(syns1, syns2, 'jcn', ic)    
        c.at[i, 'matriz_sim_lin_genesis'] = matriz_similitud(syns1, syns2, 'lin', ic)

    c.head(11)

In [ ]:
#  PUNTUACIÓN TOTAL

    # Función puntuación
        # PASO 1: Vector puntuaciones: vector de dimensión el número de filas de matriz_sim (o sea, número de elementos de syns1)
            # En cada componente se escribe la similitud máxima el elemento de syns1 y los elementos del syns2.
        # PASO 2: Puntuacion: media de todos los scores.

        
def puntuacion(matriz_sim, print_puntuacion=False):
    
    puntuacion = matriz_sim.mean()
    
    if print_puntuacion == True:
        print('PUNTUACIONES:', puntuaciones)
        print('MEDIA:', puntuacion)
    
    return(puntuacion)

# Ejemplo para comprobar funcionamiento puntuacion()

matriz_sim = c['matriz_sim_ps'][0]
print_puntuacion = False
                     
print('****** EJEMPLO puntuación() ********')   
print('matriz_sim:')
print(matriz_sim)
print('puntuacion(matriz_sim):', puntuacion(matriz_sim, print_puntuacion=print_puntuacion))
print('****')
print()


In [ ]:
# Cálculo y almacenamiento de la puntuación


if metodo_frase == 'Ariadna':

    c['puntuacion_ps'] = 0.0
    c['puntuacion_lch'] = 0.0
    c['puntuacion_wup'] = 0.0
    c['puntuacion_res_brown'] = 0.0
    c['puntuacion_res_semcor'] = 0.0
    c['puntuacion_res_genesis'] = 0.0
    c['puntuacion_jcn_brown'] = 0.0
    c['puntuacion_jcn_semcor'] = 0.0
    c['puntuacion_jcn_genesis'] = 0.0
    c['puntuacion_lin_brown'] = 0.0
    c['puntuacion_lin_semcor'] = 0.0
    c['puntuacion_lin_genesis'] = 0.0

    print_puntuacion = False
    print_info = False

    for i in c.index:
        if print_info:
            print('FILA:', i)
            print('Path_similarity')
        if c.at[i, 'matriz_sim_ps'].size != 0:
            c.at[i, 'puntuacion_ps'] = puntuacion(c.at[i, 'matriz_sim_ps'], print_puntuacion = print_puntuacion)
            if print_info: print('Puntuación', c.at[i, 'puntuacion_ps'])
        if c.at[i, 'matriz_sim_lch'].size != 0:
            if print_info: print('Leacock-Chodorow')
            c.at[i, 'puntuacion_lch'] = puntuacion(c.at[i, 'matriz_sim_lch'], print_puntuacion = print_puntuacion)
            if print_info: print('Puntuación', c.at[i, 'puntuacion_lch'])
        if c.at[i, 'matriz_sim_wup'].size != 0:
            if print_info:print('Wu-Palmer')
            c.at[i, 'puntuacion_wup'] = puntuacion(c.at[i, 'matriz_sim_wup'], print_puntuacion = print_puntuacion)
            if print_info: print('Puntuación', c.at[i, 'puntuacion_wup'])

        if c.at[i, 'matriz_sim_res_brown'].size != 0:
            if print_info: print('Resnik (Brown)')
            c.at[i, 'puntuacion_res_brown'] = puntuacion(c.at[i, 'matriz_sim_res_brown'], print_puntuacion = print_puntuacion)
            if print_info: print('Puntuación', c.at[i, 'puntuacion_res_brown'])
        if c.at[i, 'matriz_sim_jcn_brown'].size != 0:
            if print_info: print('Jiang-Conrath (Brown)')
            c.at[i, 'puntuacion_jcn_brown'] = puntuacion(c.at[i, 'matriz_sim_jcn_brown'], print_puntuacion = print_puntuacion)
            if print_info: print('Puntuación', c.at[i, 'puntuacion_jcn_brown'])
        if c.at[i, 'matriz_sim_lin_brown'].size != 0:
            if print_info: print('Lin (Brown)')
            c.at[i, 'puntuacion_lin_brown'] = puntuacion(c.at[i, 'matriz_sim_lin_brown'], print_puntuacion = print_puntuacion)
            if print_info: print('Puntuación', c.at[i, 'puntuacion_lin_brown'])
            if print_info: print()

        if c.at[i, 'matriz_sim_res_semcor'].size != 0:
            if print_info: print('Resnik (semcor)')
            c.at[i, 'puntuacion_res_semcor'] = puntuacion(c.at[i, 'matriz_sim_res_semcor'], print_puntuacion = print_puntuacion)
            if print_info: print('Puntuación', c.at[i, 'puntuacion_res_semcor'])
        if c.at[i, 'matriz_sim_jcn_semcor'].size != 0:
            if print_info: print('Jiang-Conrath (semcor)')
            c.at[i, 'puntuacion_jcn_semcor'] = puntuacion(c.at[i, 'matriz_sim_jcn_semcor'], print_puntuacion = print_puntuacion)
            if print_info: print('Puntuación', c.at[i, 'puntuacion_jcn_semcor'])
        if c.at[i, 'matriz_sim_lin_semcor'].size != 0:
            if print_info: print('Lin (semcor)')
            c.at[i, 'puntuacion_lin_semcor'] = puntuacion(c.at[i, 'matriz_sim_lin_semcor'], print_puntuacion = print_puntuacion)
            if print_info: print('Puntuación', c.at[i, 'puntuacion_lin_semcor'])
            if print_info: print()

        if c.at[i, 'matriz_sim_res_genesis'].size != 0:
            if print_info: print('Resnik (genesis)')
            c.at[i, 'puntuacion_res_genesis'] = puntuacion(c.at[i, 'matriz_sim_res_genesis'], print_puntuacion = print_puntuacion)
            if print_info: print('Puntuación', c.at[i, 'puntuacion_res_genesis'])
        if c.at[i, 'matriz_sim_jcn_genesis'].size != 0:
            if print_info: print('Jiang-Conrath (genesis)')
            c.at[i, 'puntuacion_jcn_genesis'] = puntuacion(c.at[i, 'matriz_sim_jcn_genesis'], print_puntuacion = print_puntuacion)
            if print_info: print('Puntuación', c.at[i, 'puntuacion_jcn_genesis'])
        if c.at[i, 'matriz_sim_lin_genesis'].size != 0:
            if print_info: print('Lin (genesis)')
            c.at[i, 'puntuacion_lin_genesis'] = puntuacion(c.at[i, 'matriz_sim_lin_genesis'], print_puntuacion = print_puntuacion)
            if print_info: print('Puntuación', c.at[i, 'puntuacion_lin_genesis'])
            if print_info: print()

end_total = time.time()

print('TIEMPO TRANSCURRIDO:', end_total-start_total)

### Results export

The resulting dataframe is exported to an Excel sheet for further analysis. Pearson correlation for each experiment is computed in [this Excel Worksheet]. Graphs are also provided for easy comparison.

In [ ]:
# Fix export problems with illegal characters in Excel 
import re

ILLEGAL_CHARACTERS_RE = re.compile(r'[\x00-\x08\x0B-\x0C\x0E-\x1F]')

def clean_illegal_chars(value):
    if isinstance(value, str):
        return ILLEGAL_CHARACTERS_RE.sub('', value)
    return value

c_export = c.apply(lambda col: col.map(clean_illegal_chars))

x = datetime.datetime.now()
c_export.to_excel(x.strftime("%y%m%d-%H%M-") + criterios + '.xlsx')